# 01 — Extraction GDELT · Phase 2 · Bénin Pulse

**Objectif :** alimenter un dashboard multi-profils (investisseur, opérateur économique, diaspora).

## Corrections majeures vs Phase 1 et Phase 2

| Problème | Correction |
|---|---|
| Filtre Nigeria incomplet (seulement 'Benin City') | Exclusion par CountryCode NI + bounding box GPS |
| Pas de contrôle par coordonnées GPS | Filtre lat [6.1, 12.4] et lon [0.8, 3.85] |
| Pas de comparaison médias locaux vs internationaux | `source_type` dérivé du domaine SOURCEURL |
| Pas de table dédiée aux événements économiques | Nouvelle table `benin_eco_events.csv` |
| Pas de table relations bilatérales | Nouvelle table `benin_bilateral.csv` |
| Pas de table pour biais médiatique | Nouvelle table `benin_media_bias.csv` |

## Tables produites
1. `benin_events_clean.csv` — événements filtrés et enrichis (base principale)
2. `comparatif_regional.csv` — Bénin vs 6 pays voisins
3. `benin_eco_events.csv` — événements économiques positifs
4. `benin_bilateral.csv` — relations bilatérales Bénin-voisins
5. `benin_media_bias.csv` — médias locaux vs internationaux
6. `benin_sector_themes.csv` — radar sectoriel depuis GKG
7. `benin_villes_gkg.csv` — carte de chaleur par ville/département

## 0. Configuration et connexion

In [3]:
from google.colab import auth, drive
auth.authenticate_user()

from google.cloud import bigquery
from datetime import date, timedelta
import pandas as pd
import os, re

project_id = 'hackathon-benin-insight'
client = bigquery.Client(project=project_id)

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/data/raw'
LOCAL_PATH = '/content/data/raw'
os.makedirs(DRIVE_PATH, exist_ok=True)
os.makedirs(LOCAL_PATH, exist_ok=True)

# Fenetre glissante 12 mois
DATE_FIN       = date.today()
DATE_DEBUT     = DATE_FIN - timedelta(days=365)
DATE_FIN_STR   = DATE_FIN.strftime('%Y-%m-%d')
DATE_DEBUT_STR = DATE_DEBUT.strftime('%Y-%m-%d')

# ── Bounding box GPS du Benin ──────────────────────────────────────────────
# Source : Natural Earth / OpenStreetMap
# Tout point hors de cette boite et classe BN est une erreur de geocodage GDELT
LAT_MIN, LAT_MAX =  6.10, 12.42
LON_MIN, LON_MAX =  0.77,  3.85

# ── Domaines de medias beninois (pour classification locale) ───────────────
MEDIAS_LOCAUX = [
    # déjà présents
    'banouto.bj', 'lanation.bj', 'fraternite.bj', 'acotonou.com',
    'beninwebtv.com', 'lepatriote.net', 'matin.bj',
    'beninactu.net', 'nouvelles-du-benin.com', 'soleil.bj',
    'abpbenin.com',  # Agence Bénin Presse officielle
    'gouv.bj', 'ortb.bj',
]

MEDIAS_REGIONAUX = [
    'allafrica.com', 'jeuneafrique.com', 'rfi.fr', 'bbc.com/afrique',
    'voaafrique.com', 'africanews.com', 'agenceecofin.com', 'afriquexxi.info'
]

# Mapping FIPS -> nom lisible
NOMS_PAYS = {
    'BN': 'Benin', 'TO': 'Togo', 'CM': 'Cameroun', 'GH': 'Ghana',
    'NI': 'Nigeria', 'NG': 'Niger', 'UV': 'Burkina Faso'
}

print(f'Connected — project: {client.project}')
print(f'Window: {DATE_DEBUT_STR} -> {DATE_FIN_STR}')
print(f'Bounding box: lat [{LAT_MIN}, {LAT_MAX}] | lon [{LON_MIN}, {LON_MAX}]')

def sauvegarder(df, nom):
    df.to_csv(f'{LOCAL_PATH}/{nom}', index=False)
    df.to_csv(f'{DRIVE_PATH}/{nom}', index=False)
    print(f'OK {nom} -> {len(df):,} lignes, {df.shape[1]} colonnes')

def classifier_source(url):
    if not isinstance(url, str):
        return 'inconnu'
    url_lower = url.lower()
    if any(d in url_lower for d in MEDIAS_LOCAUX):
        return 'local'
    if any(d in url_lower for d in MEDIAS_REGIONAUX):
        return 'regional'
    return 'international'

def extraire_domaine(url):
    if not isinstance(url, str):
        return ''
    match = re.search(r'https?://([^/]+)', url)
    return match.group(1).replace('www.', '') if match else ''

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Connected — project: hackathon-benin-insight
Window: 2025-05-28 -> 2026-05-28
Bounding box: lat [6.1, 12.42] | lon [0.77, 3.85]


## 1. Macro filtre Benin — definition officielle

Ce bloc SQL est injecte dans toutes les requetes suivantes.

**Quatre niveaux de protection contre la contamination Nigeria :**
1. `ActionGeo_CountryCode != 'NI'` — code pays
2. Bounding box GPS — lat/lon dans les frontières réelles du Bénin
3. Liste noire de toponymes nigérians connus
4. Contrôle Python post-extraction en double vérification

In [13]:
FILTRE_BENIN = f"""
  -- 1. Partition en premier -> economie de quota
  _PARTITIONDATE BETWEEN '{DATE_DEBUT_STR}' AND '{DATE_FIN_STR}'

  -- 2. Critere d'inclusion : geographie OU acteur beninois
  AND (
    (
      ActionGeo_CountryCode = 'BN'
      AND ActionGeo_Lat  BETWEEN {LAT_MIN} AND {LAT_MAX}
      AND ActionGeo_Long BETWEEN {LON_MIN} AND {LON_MAX}
    )
    OR Actor1CountryCode = 'BEN'
    OR Actor2CountryCode = 'BEN'
  )

  -- 3. Exclusion Nigeria multicouche
  AND ActionGeo_CountryCode != 'NI'
  AND ActionGeo_FullName NOT LIKE '%Nigeria%'
  AND ActionGeo_FullName NOT LIKE '%Benin City%'
  AND ActionGeo_FullName NOT LIKE '%Edo State%'
  AND ActionGeo_FullName NOT LIKE '%Ondo%'
  AND ActionGeo_FullName NOT LIKE '%Kogi%'
  AND ActionGeo_FullName NOT LIKE '%Sapele%'
  AND ActionGeo_FullName NOT LIKE '%Warri%'

  -- 4. Exclusion autres pays voisins mal geocodes
  AND ActionGeo_FullName NOT LIKE '%Togo%'
  AND ActionGeo_FullName NOT LIKE '%Niger,%'
  AND ActionGeo_FullName NOT LIKE '%Burkina%'
"""

print('Macro filtre defini - pret pour injection dans toutes les requetes')

Macro filtre defini - pret pour injection dans toutes les requetes


## 2. Validation — Vérification du filtre geographique

Toujours lancer cette cellule en premier. Elle confirme que:
- Le code BN retourne bien des lieux beninois
- Aucun lieu nigerien ne passe le bounding box GPS
- Le contrôle automatique Python compte 0 anomalie

In [14]:
query_validation = f"""
SELECT
  ActionGeo_CountryCode,
  ActionGeo_FullName,
  ActionGeo_Lat,
  ActionGeo_Long,
  COUNT(*) AS nb
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '{DATE_DEBUT_STR}' AND '{DATE_FIN_STR}'
  AND ActionGeo_CountryCode = 'BN'
  AND ActionGeo_Lat  BETWEEN {LAT_MIN} AND {LAT_MAX}
  AND ActionGeo_Long BETWEEN {LON_MIN} AND {LON_MAX}
  AND ActionGeo_CountryCode != 'NI'
  AND ActionGeo_FullName NOT LIKE '%Nigeria%'
  AND ActionGeo_FullName NOT LIKE '%Benin City%'
  AND ActionGeo_FullName NOT LIKE '%Edo State%'
GROUP BY 1, 2, 3, 4
ORDER BY nb DESC
LIMIT 40
"""

print('Validation du filtre geographique...')
df_validation = client.query(query_validation).to_dataframe()

# Controle automatique
hors_bbox = df_validation[
    (df_validation['ActionGeo_Lat'] < LAT_MIN) |
    (df_validation['ActionGeo_Lat'] > LAT_MAX) |
    (df_validation['ActionGeo_Long'] < LON_MIN) |
    (df_validation['ActionGeo_Long'] > LON_MAX)
]
print(f'Lieux hors bounding box Benin : {len(hors_bbox)} (doit etre 0)')
if len(hors_bbox) > 0:
    print('ANOMALIES DETECTEES :')
    print(hors_bbox[['ActionGeo_FullName', 'ActionGeo_Lat', 'ActionGeo_Long']])
else:
    print('OK - Aucune anomalie geographique')

display(df_validation.head(20))

Validation du filtre geographique...
Lieux hors bounding box Benin : 0 (doit etre 0)
OK - Aucune anomalie geographique


,ActionGeo_CountryCode,ActionGeo_FullName,ActionGeo_Lat,ActionGeo_Long,nb
0,BN,Benin,9.50000,2.250000,20430
1,BN,"Porto-Novo, Qué, Benin",6.48333,2.616670,243
2,BN,"Ouidah, Atlantique, Benin",6.36307,2.085060,189
3,BN,"Abomey, Zou, Benin",7.18286,1.991190,102
4,BN,"Lokossa, Mono, Benin",6.63869,1.716740,87
5,BN,"Tchaourou, Benin (general), Benin",8.88649,2.597520,87
6,BN,"Porga, Atakora, Benin",11.04800,0.968912,66
7,BN,"Malanville, Atakora, Benin",11.86850,3.389890,48
8,BN,"Couffo, Kouffo, Benin",7.08333,1.833330,41
9,BN,"Kandi, Alibori, Benin",11.13420,2.938610,41


## 3. Table principale — benin_events_clean.csv

Nouveautes vs Phase 2:
- Bounding box GPS integre dans le WHERE
- `source_type` : local / regional / international base sur le domaine SOURCEURL
- `source_domain` : domaine extrait proprement
- `periode` : annotation automatique des periodes de crise
- `semaine` : agregation temporelle pour les graphiques

In [15]:
PERIODES_CRISE = [
    {'label': 'Attaque_Alibori_Avr2025',  'debut': 20250415, 'fin': 20250430},
    {'label': 'CoupEtat_Dec2025',          'debut': 20251205, 'fin': 20251215},
    {'label': 'Attaque_Kofouno_Mar2026',   'debut': 20260303, 'fin': 20260310},
    {'label': 'Election_Presidentielle',   'debut': 20260415, 'fin': 20260430},
]

def annoter_periode(sqldate):
    for p in PERIODES_CRISE:
        if p['debut'] <= sqldate <= p['fin']:
            return p['label']
    return 'Periode_normale'

query_events = f"""
SELECT
  GLOBALEVENTID, SQLDATE, DATEADDED,
  QuadClass, IsRootEvent,
  EventCode, EventBaseCode, EventRootCode,
  Actor1Name, Actor1CountryCode, Actor1Type1Code,
  Actor2Name, Actor2CountryCode, Actor2Type1Code,
  GoldsteinScale, NumMentions, NumSources, NumArticles, AvgTone,
  ActionGeo_Type, ActionGeo_FullName, ActionGeo_CountryCode,
  ActionGeo_Lat, ActionGeo_Long, ActionGeo_FeatureID,
  SOURCEURL
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE {FILTRE_BENIN}
ORDER BY SQLDATE
"""

print('Extraction events en cours...')
df_events = client.query(query_events).to_dataframe()
print(f'Brut : {len(df_events):,} lignes')

# Deduplication
df_events = (
    df_events
    .sort_values('NumArticles', ascending=False)
    .drop_duplicates(subset=['GLOBALEVENTID'], keep='first')
    .sort_values('SQLDATE')
    .reset_index(drop=True)
)
print(f'Apres deduplication : {len(df_events):,} lignes')

# Nettoyage lignes vides
mask_vide = (
    df_events['ActionGeo_FullName'].isna() &
    df_events['Actor1Name'].isna() &
    df_events['Actor2Name'].isna()
)
df_events = df_events[~mask_vide].reset_index(drop=True)
print(f'Apres nettoyage : {len(df_events):,} lignes')

# Enrichissements
geo_labels = {1:'pays', 2:'etat_usa', 3:'ville_usa', 4:'region', 5:'ville'}
df_events['geo_precision'] = df_events['ActionGeo_Type'].map(geo_labels).fillna('inconnu')
df_events['source_type']   = df_events['SOURCEURL'].apply(classifier_source)
df_events['source_domain'] = df_events['SOURCEURL'].apply(extraire_domaine)
df_events['periode']       = df_events['SQLDATE'].apply(annoter_periode)
df_events['semaine']       = pd.to_datetime(
    df_events['SQLDATE'].astype(str), format='%Y%m%d'
).dt.to_period('W').astype(str)

# Controle final contamination Nigeria
contamination = df_events[
    df_events['ActionGeo_FullName'].str.contains('Nigeria|Benin City|Edo State', na=False, case=False)
]
print(f'\nContamination Nigeria residuelle : {len(contamination)} lignes (doit etre 0)')

print('\nRepartition sources :')
print(df_events['source_type'].value_counts().to_string())
print('\nPrecision geographique :')
print(df_events['geo_precision'].value_counts().to_string())

sauvegarder(df_events, 'benin_events_clean.csv')

Extraction events en cours...
Brut : 25,629 lignes
Apres deduplication : 25,629 lignes
Apres nettoyage : 25,629 lignes

Contamination Nigeria residuelle : 0 lignes (doit etre 0)

Repartition sources :
source_type
international    24204
regional          1231
local              194

Precision geographique :
geo_precision
pays         22326
region        2983
ville          195
ville_usa       86
etat_usa        39
OK benin_events_clean.csv -> 25,629 lignes, 31 colonnes


## 4. Table — benin_media_bias.csv

**Hypothèse à valider :**
Les medias internationaux ont-ils dramatise la situation lors du coup d'Etat
du 7 decembre 2025 et des attaques dans le nord, par rapport a la realite
vecue par les Beninois ?

**Methode :** comparaison AvgTone et GoldsteinScale par type de source
sur les periodes de crise annotees.

**Resultat attendu :** un ecart negatif (international < local) sur AvgTone
= preuve chiffree que les medias internationaux ont ete plus pessimistes
que les medias beninois sur la meme periode.

In [16]:
# Agregation semaine x source_type x periode
df_bias = (
    df_events
    .groupby(['semaine', 'source_type', 'periode'])
    .agg(
        nb_evenements   = ('GLOBALEVENTID', 'count'),
        ton_moyen       = ('AvgTone', 'mean'),
        goldstein_moyen = ('GoldsteinScale', 'mean'),
        nb_articles     = ('NumArticles', 'sum'),
        pct_conflits    = ('QuadClass', lambda x: round(x.isin([3,4]).sum() / len(x) * 100, 2))
    )
    .reset_index()
    .sort_values(['semaine', 'source_type'])
)

# Resume : distorsion mediatique par periode de crise
print('Comparaison ton moyen par type de source sur les periodes de crise:')
print('=' * 70)
crises = df_bias[df_bias['periode'] != 'Periode_normale']
if len(crises) > 0:
    pivot = crises.pivot_table(
        index='periode',
        columns='source_type',
        values=['ton_moyen', 'goldstein_moyen', 'nb_articles'],
        aggfunc='mean'
    ).round(2)
    print(pivot.to_string())
    print('\nEcart positif = medias internationaux plus negatifs que medias locaux')
else:
    print('Aucune periode de crise dans la fenetre de donnees')

sauvegarder(df_bias, 'benin_media_bias.csv')

Comparaison ton moyen par type de source sur les periodes de crise:
                        goldstein_moyen                  nb_articles                    ton_moyen               
source_type               international local regional international local regional international local regional
periode                                                                                                         
Attaque_Kofouno_Mar2026            0.10  4.44     2.36        1175.0  90.0     91.5         -1.59  5.38    -1.99
CoupEtat_Dec2025                   0.23  2.63    -1.60       6357.67  25.0   369.33         -2.47  6.87    -3.48
Election_Presidentielle            0.08  2.33    -0.35       1789.67  85.0    95.67         -2.41  3.09    -4.18

Ecart positif = medias internationaux plus negatifs que medias locaux
OK benin_media_bias.csv -> 151 lignes, 8 colonnes


## 5. Table — comparatif_regional.csv

Benin vs Togo, Cameroun, Ghana, Nigeria, Niger, Burkina Faso.
Metriques mensuelles pour le classement de stabilite et le benchmark regional.

**Codes FIPS verifies :**
BN=Benin · TO=Togo · CM=Cameroun · GH=Ghana · NI=Nigeria · NG=Niger · UV=Burkina

In [17]:
query_regional = f"""
SELECT
  ActionGeo_CountryCode AS fips_pays,
  EXTRACT(YEAR  FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS annee,
  EXTRACT(MONTH FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS mois,
  COUNT(*)                                               AS nb_evenements,
  SUM(NumArticles)                                       AS nb_articles_total,
  AVG(AvgTone)                                           AS ton_moyen,
  AVG(GoldsteinScale)                                    AS goldstein_moyen,
  STDDEV(AvgTone)                                        AS ton_volatilite,
  COUNTIF(QuadClass IN (3,4))                            AS nb_conflits,
  ROUND(COUNTIF(QuadClass IN (3,4))/COUNT(*)*100,2)     AS pct_conflits,
  COUNTIF(QuadClass IN (1,2))                            AS nb_cooperation,
  ROUND(COUNTIF(QuadClass IN (1,2))/COUNT(*)*100,2)     AS pct_cooperation,
  AVG(NumMentions)                                       AS mentions_moyennes
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '{DATE_DEBUT_STR}' AND '{DATE_FIN_STR}'
  AND ActionGeo_CountryCode IN ('BN','TO','CM','GH','NI','NG','UV')
GROUP BY fips_pays, annee, mois
ORDER BY annee, mois, fips_pays
"""

print('Extraction comparatif regional...')
df_regional = client.query(query_regional).to_dataframe()
df_regional['pays_nom'] = df_regional['fips_pays'].map(NOMS_PAYS)
print(f'{len(df_regional):,} lignes (pays x mois)')

print('\nClassement stabilite (GoldsteinScale moyen) :')
classement = (
    df_regional.groupby('pays_nom')[['goldstein_moyen','ton_moyen','pct_conflits']]
    .mean().round(2)
    .sort_values('goldstein_moyen', ascending=False)
)
print(classement.to_string())

sauvegarder(df_regional, 'comparatif_regional.csv')

Extraction comparatif regional...
91 lignes (pays x mois)

Classement stabilite (GoldsteinScale moyen) :
              goldstein_moyen  ton_moyen  pct_conflits
pays_nom                                              
Togo                     1.40      -0.73         21.22
Ghana                    1.18      -0.19         20.10
Cameroun                 0.89      -1.94         23.45
Burkina Faso             0.78      -1.52         26.30
Benin                    0.60      -1.46         25.57
Nigeria                  0.22      -1.69         28.70
Niger                   -0.11      -2.90         32.23
OK comparatif_regional.csv -> 91 lignes, 14 colonnes


## 6. Table — benin_eco_events.csv

Pour le profil Investisseur et Operateur economique.
Evenements de cooperation economique impliquant le Benin :
accords, investissements, visites diplomatiques, partenariats.

EventRootCode 03-08 = cooperation, consultation, negociation, accord.

In [18]:
query_eco = f"""
SELECT
  GLOBALEVENTID, SQLDATE,
  Actor1Name, Actor1CountryCode,
  Actor2Name, Actor2CountryCode,
  EventCode, EventBaseCode, EventRootCode,
  QuadClass, GoldsteinScale, AvgTone, NumArticles,
  ActionGeo_FullName, ActionGeo_Lat, ActionGeo_Long,
  SOURCEURL
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  {FILTRE_BENIN}
  AND QuadClass IN (1, 2)
  AND EventRootCode IN ('04','05','06')  -- Consulter, Engager, Cooperer (retire 03/07/08 trop larges)
  AND GoldsteinScale > 0                 -- Uniquement evenements a connotation positive
ORDER BY SQLDATE, NumArticles DESC
"""

print('Extraction evenements economiques...')
df_eco = client.query(query_eco).to_dataframe()
df_eco['source_type']   = df_eco['SOURCEURL'].apply(classifier_source)
df_eco['source_domain'] = df_eco['SOURCEURL'].apply(extraire_domaine)

print(f'{len(df_eco):,} evenements economiques positifs')
print('\nTop partenaires du Benin :')
partenaires = pd.concat([
    df_eco[df_eco['Actor1CountryCode'] != 'BEN']['Actor1CountryCode'],
    df_eco[df_eco['Actor2CountryCode'] != 'BEN']['Actor2CountryCode']
]).value_counts().head(15)
print(partenaires.to_string())

sauvegarder(df_eco, 'benin_eco_events.csv')

Extraction evenements economiques...
10,079 evenements economiques positifs

Top partenaires du Benin :
NGA    1619
AFR     662
FRA     382
GHA     211
WAF     209
NER     193
USA     190
GBR     168
CHN     158
TGO     143
BFA     121
CIV     104
SEN     101
RUS      84
KEN      82
OK benin_eco_events.csv -> 10,079 lignes, 19 colonnes


## 7. Table — benin_bilateral.csv

Relations bilaterales Benin avec ses voisins, mois par mois.
GoldsteinScale positif = cooperation | negatif = tension.
Utile pour : Benin-Nigeria (flux commerciaux), Benin-Togo (integration regionale).

In [19]:
query_bilateral = f"""
SELECT
  EXTRACT(YEAR  FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS annee,
  EXTRACT(MONTH FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS mois,
  CASE
    WHEN Actor1CountryCode = 'BEN' THEN Actor2CountryCode
    ELSE Actor1CountryCode
  END AS pays_partenaire,
  COUNT(*)                        AS nb_interactions,
  AVG(GoldsteinScale)             AS qualite_relation,
  AVG(AvgTone)                    AS ton_moyen,
  COUNTIF(QuadClass IN (1,2))     AS nb_cooperation,
  COUNTIF(QuadClass IN (3,4))     AS nb_conflits,
  SUM(NumArticles)                AS visibilite_mediatique
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  {FILTRE_BENIN}
  AND (
    (Actor1CountryCode = 'BEN' AND Actor2CountryCode IN ('NGA','TGO','NER','GHA','BFA','CIV','SEN','FRA','CHN'))
    OR
    (Actor2CountryCode = 'BEN' AND Actor1CountryCode IN ('NGA','TGO','NER','GHA','BFA','CIV','SEN','FRA','CHN'))
  )
  AND Actor1CountryCode IS NOT NULL
  AND Actor2CountryCode IS NOT NULL
GROUP BY annee, mois, pays_partenaire
ORDER BY annee, mois, pays_partenaire
"""

# Noms lisibles pour l'affichage dashboard
NOMS_PARTENAIRES = {
    'NGA': 'Nigeria', 'TGO': 'Togo', 'NER': 'Niger',
    'GHA': 'Ghana',   'BFA': 'Burkina Faso', 'CIV': "Cote d'Ivoire",
    'SEN': 'Senegal', 'FRA': 'France', 'CHN': 'Chine'
}

print('Extraction relations bilaterales...')
df_bilateral = client.query(query_bilateral).to_dataframe()
df_bilateral['pays_nom'] = df_bilateral['pays_partenaire'].map(NOMS_PARTENAIRES).fillna(df_bilateral['pays_partenaire'])
print(f'{len(df_bilateral):,} lignes (partenaire x mois)')

print('\nQualite des relations bilaterales (GoldsteinScale moyen) :')
print('Positif = cooperation | Negatif = tension')
resume = (
    df_bilateral.groupby('pays_nom')
    [['qualite_relation','ton_moyen','nb_cooperation','nb_conflits']]
    .mean().round(2)
    .sort_values('qualite_relation', ascending=False)
)
print(resume.to_string())

sauvegarder(df_bilateral, 'benin_bilateral.csv')

Extraction relations bilaterales...
104 lignes (partenaire x mois)

Qualite des relations bilaterales (GoldsteinScale moyen) :
Positif = cooperation | Negatif = tension
               qualite_relation  ton_moyen  nb_cooperation  nb_conflits
pays_nom                                                               
Ghana                      2.77       0.12           14.08          3.0
Chine                      2.54       2.81           12.23         1.08
Senegal                    1.42       0.30             9.2          1.5
Nigeria                    0.84      -0.55            81.0        29.15
France                     0.60      -1.42           30.92        10.17
Cote d'Ivoire             -0.38      -4.07            7.38         3.12
Togo                      -0.42      -1.80            8.92         4.85
Niger                     -1.15      -4.02            13.0         17.0
Burkina Faso              -1.44      -2.64             5.3          5.7
OK benin_bilateral.csv -> 104 lignes, 1

## 8. Table — benin_sector_themes.csv depuis GKG

Radar sectoriel : quels secteurs sont positivement ou negativement couverts ?
Parsing de V2Themes du GKG pour detecter :
Securite, Economie, Agriculture, Tourisme, Politique, Sante, Environnement.

In [20]:
SECTEURS = {
    'Securite'      : ['TERROR', 'MILITARY', 'INSURGENCY', 'ARMED_CONFLICT'],
    'Economie'      : ['ECON_', 'TRADE', 'INVESTMENT', 'BUSINESS', 'MARKET'],
    'Agriculture'   : ['FOOD_', 'AGRICULTURE', 'CROP', 'FAMINE', 'DROUGHT'],
    'Politique'     : ['GOV_', 'ELECTION_', 'DEMOCRACY', 'POLITICAL', 'COUP'],
    'Tourisme'      : ['TOURISM', 'CULTURE', 'UNESCO', 'HERITAGE'],
    'Sante'         : ['HEALTH_', 'DISEASE_', 'MEDICAL', 'PANDEMIC'],
    'Education'     : ['EDUCATION_', 'SCHOOL', 'UNIVERSITY'],
    'Environnement' : ['ENV_', 'CLIMATE', 'FLOOD'],
}

def detecter_secteurs(themes_str):
    if not isinstance(themes_str, str):
        return ['Autre']
    themes_upper = themes_str.upper()
    detectes = [s for s, mots in SECTEURS.items() if any(m in themes_upper for m in mots)]
    return detectes if detectes else ['Autre']

# Chargement GKG depuis CSV local si disponible
try:
    df_gkg = pd.read_csv(f'{LOCAL_PATH}/benin_gkg.csv')
    print(f'GKG charge depuis CSV local : {len(df_gkg):,} lignes')
except FileNotFoundError:
    print('GKG non trouve localement - extraction BigQuery...')
    query_gkg = f"""
    SELECT DATE, DocumentIdentifier, V2Themes, V2Tone
    FROM `gdelt-bq.gdeltv2.gkg_partitioned`
    WHERE
      _PARTITIONDATE BETWEEN '{DATE_DEBUT_STR}' AND '{DATE_FIN_STR}'
      AND V2Locations LIKE '%#BN#%'
    """
    df_gkg = client.query(query_gkg).to_dataframe()
    df_gkg.to_csv(f'{LOCAL_PATH}/benin_gkg.csv', index=False)
    df_gkg.to_csv(f'{DRIVE_PATH}/benin_gkg.csv', index=False)
    print(f'GKG extrait et sauvegarde : {len(df_gkg):,} lignes')

# Parsing des themes
print('Parsing des secteurs depuis V2Themes...')
rows_secteurs = []
for _, row in df_gkg.iterrows():
    secteurs = detecter_secteurs(row.get('V2Themes', ''))
    tone_parts = str(row.get('V2Tone', '')).split(',')
    try:
        tone = float(tone_parts[0])
    except (ValueError, IndexError):
        tone = None
    date_val = str(row.get('DATE', ''))[:8]
    for secteur in secteurs:
        rows_secteurs.append({'date': date_val, 'secteur': secteur, 'tone': tone})

df_secteurs_raw = pd.DataFrame(rows_secteurs)
df_secteurs_raw['mois'] = df_secteurs_raw['date'].str[:6]

df_themes = (
    df_secteurs_raw
    .groupby(['mois', 'secteur'])
    .agg(nb_mentions=('tone','count'), tone_moyen=('tone','mean'))
    .reset_index()
    .sort_values(['mois', 'nb_mentions'], ascending=[True, False])
)

print('\nTop secteurs sur 12 mois :')
top = (
    df_secteurs_raw.groupby('secteur')
    .agg(nb_mentions=('tone','count'), tone_moyen=('tone','mean'))
    .round(2)
    .sort_values('nb_mentions', ascending=False)
)
print(top.to_string())

sauvegarder(df_themes, 'benin_sector_themes.csv')

GKG charge depuis CSV local : 47,449 lignes
Parsing des secteurs depuis V2Themes...

Top secteurs sur 12 mois :
               nb_mentions  tone_moyen
secteur                               
Economie             19653       -0.55
Politique            15837       -1.95
Securite             14704       -2.80
Sante                13804       -0.98
Autre                12304        0.55
Education            10709       -0.11
Environnement         9466       -0.78
Tourisme              8369        0.06
Agriculture           6940       -0.35
OK benin_sector_themes.csv -> 117 lignes, 4 colonnes


## 9. Verification finale

In [21]:
print('=' * 65)
print('RESUME EXTRACTION — Benin Pulse')
print('=' * 65)

fichiers = {
    'benin_events_clean.csv'  : df_events,
    'comparatif_regional.csv' : df_regional,
    'benin_eco_events.csv'    : df_eco,
    'benin_bilateral.csv'     : df_bilateral,
    'benin_media_bias.csv'    : df_bias,
    'benin_sector_themes.csv' : df_themes,
}

for nom, df in fichiers.items():
    taille = df.memory_usage(deep=True).sum() / 1024**2
    print(f'  {nom:<35} {len(df):>8,} lignes  | {taille:>5.1f} MB')

print('\n-- Controle contamination Nigeria --')
nb_ni = df_events[
    df_events['ActionGeo_FullName'].str.contains('Nigeria|Benin City|Edo State', na=False, case=False)
]
print(f'  Lignes nigerianes residuelles : {len(nb_ni)} (doit etre 0)')

print('\n-- Couverture temporelle --')
dates = pd.to_datetime(df_events['SQLDATE'].astype(str), format='%Y%m%d')
print(f'  Debut : {dates.min().date()}')
print(f'  Fin   : {dates.max().date()}')
print(f'  Jours : {(dates.max() - dates.min()).days}')

print('\n-- Repartition sources --')
print(df_events['source_type'].value_counts().to_string())

print('\nExtraction terminee. Ne relancer BigQuery que pour le rafraichissement hebdomadaire.')

RESUME EXTRACTION — Benin Pulse
  benin_events_clean.csv                25,629 lignes  |  27.8 MB
  comparatif_regional.csv                   91 lignes  |   0.0 MB
  benin_eco_events.csv                  10,079 lignes  |   7.1 MB
  benin_bilateral.csv                      104 lignes  |   0.0 MB
  benin_media_bias.csv                     151 lignes  |   0.0 MB
  benin_sector_themes.csv                  117 lignes  |   0.0 MB

-- Controle contamination Nigeria --
  Lignes nigerianes residuelles : 0 (doit etre 0)

-- Couverture temporelle --
  Debut : 2025-05-28
  Fin   : 2026-05-27
  Jours : 364

-- Repartition sources --
source_type
international    24204
regional          1231
local              194

Extraction terminee. Ne relancer BigQuery que pour le rafraichissement hebdomadaire.


In [22]:
# Quels codes pays GDELT utilise-t-il réellement pour les voisins du Bénin ?
query_codes = f"""
SELECT
  Actor1CountryCode,
  Actor2CountryCode,
  COUNT(*) as nb
FROM `gdelt-bq.gdeltv2.events_partitioned`
WHERE
  _PARTITIONDATE BETWEEN '{DATE_DEBUT_STR}' AND '{DATE_FIN_STR}'
  AND (Actor1CountryCode = 'BEN' OR Actor2CountryCode = 'BEN')
  AND Actor1CountryCode IS NOT NULL
  AND Actor2CountryCode IS NOT NULL
GROUP BY 1, 2
ORDER BY nb DESC
LIMIT 30
"""
df_codes = client.query(query_codes).to_dataframe()
display(df_codes)

,Actor1CountryCode,Actor2CountryCode,nb
0,NGA,BEN,1681
1,BEN,NGA,1328
2,BEN,BEN,482
3,BEN,AFR,333
4,FRA,BEN,299
5,AFR,BEN,274
6,NER,BEN,244
7,BEN,FRA,223
8,WAF,BEN,176
9,BEN,NER,166


In [23]:
print(df_eco['EventRootCode'].value_counts().head(10))
print(df_eco['GoldsteinScale'].describe())

EventRootCode
04    6287
05    3242
06     550
Name: count, dtype: int64
count    10079.000000
mean         3.186417
std          1.788065
min          1.000000
25%          1.900000
50%          2.800000
75%          3.400000
max          8.000000
Name: GoldsteinScale, dtype: float64


In [1]:
# ── Cellule : Chargement données FMI ─────────────────────────────────────
import requests
import pandas as pd

# Indicateurs FMI utiles pour notre produit
# Codes disponibles sur : imf.org/external/datamapper/api/v1/indicator
INDICATEURS_FMI = {
    'NGDP_RPCH'  : 'croissance_pib',        # Croissance réelle du PIB (%)
    'PCPIPCH'    : 'inflation',              # Inflation (%)
    'LUR'        : 'taux_chomage',           # Chômage (%)
    'BCA_NGDPD'  : 'balance_courante_pib',  # Balance courante / PIB
    'GGXWDG_NGDP': 'dette_publique_pib',    # Dette publique / PIB
    'NGDPDPC'    : 'pib_par_habitant',      # PIB par habitant (USD)
}

# Pays à comparer (codes ISO FMI)
PAYS_FMI = ['BEN', 'TGO', 'GHA', 'NGA', 'BFA', 'NER', 'CMR']
NOMS_FMI  = {
    'BEN': 'Bénin', 'TGO': 'Togo', 'GHA': 'Ghana',
    'NGA': 'Nigeria', 'BFA': 'Burkina Faso',
    'NER': 'Niger', 'CMR': 'Cameroun'
}

def fetch_fmi(indicator_code, indicator_name):
    """Télécharge un indicateur FMI pour tous les pays cibles."""
    url = f"https://imf.org/external/datamapper/api/v1/{indicator_code}"
    try:
        r = requests.get(url, timeout=15)
        data = r.json()['values'][indicator_code]
        rows = []
        for pays in PAYS_FMI:
            if pays in data:
                for annee, valeur in data[pays].items():
                    rows.append({
                        'pays_code'  : pays,
                        'pays_nom'   : NOMS_FMI.get(pays, pays),
                        'indicateur' : indicator_name,
                        'annee'      : int(annee),
                        'valeur'     : valeur
                    })
        return pd.DataFrame(rows)
    except Exception as e:
        print(f"Erreur {indicator_code} : {e}")
        return pd.DataFrame()

print("Téléchargement données FMI...")
frames = []
for code, nom in INDICATEURS_FMI.items():
    df_ind = fetch_fmi(code, nom)
    if not df_ind.empty:
        frames.append(df_ind)
        print(f"  OK {nom} : {len(df_ind)} lignes")

df_fmi = pd.concat(frames, ignore_index=True)
print(f"\nTotal : {len(df_fmi):,} lignes")
print(f"Années disponibles : {df_fmi['annee'].min()} → {df_fmi['annee'].max()}")

Téléchargement données FMI...
  OK croissance_pib : 353 lignes
  OK inflation : 348 lignes
  OK taux_chomage : 9 lignes
  OK balance_courante_pib : 354 lignes
  OK dette_publique_pib : 246 lignes
  OK pib_par_habitant : 354 lignes

Total : 1,664 lignes
Années disponibles : 1980 → 2031


In [4]:
# ── Cellule : Nettoyage FMI ───────────────────────────────────────────────

# Garder 2018 → 2026 (inclut projections FMI)
df_fmi = df_fmi[df_fmi['annee'].between(2018, 2026)].copy()
df_fmi['valeur'] = pd.to_numeric(df_fmi['valeur'], errors='coerce')

# Pivot large : une ligne par pays × année, une colonne par indicateur
df_fmi_pivot = df_fmi.pivot_table(
    index=['pays_code', 'pays_nom', 'annee'],
    columns='indicateur',
    values='valeur'
).reset_index()
df_fmi_pivot.columns.name = None

print("=== RÉSUMÉ FMI — Bénin ===")
benin_fmi = df_fmi_pivot[df_fmi_pivot['pays_code'] == 'BEN'].sort_values('annee')
print(benin_fmi[['annee','croissance_pib','inflation','pib_par_habitant','dette_publique_pib']].to_string(index=False))

# Sauvegarde
df_fmi_pivot.to_csv(f'{LOCAL_PATH}/fmi_comparatif.csv', index=False)
df_fmi_pivot.to_csv(f'{DRIVE_PATH}/fmi_comparatif.csv', index=False)
print(f"\nOK fmi_comparatif.csv → {len(df_fmi_pivot):,} lignes")

=== RÉSUMÉ FMI — Bénin ===
 annee  croissance_pib  inflation  pib_par_habitant  dette_publique_pib
  2018             6.6        0.8          1158.452                40.8
  2019             7.1       -0.9          1154.817                40.4
  2020             3.8        3.0          1199.234                46.1
  2021             7.2        1.7          1319.528                55.6
  2022             6.3        1.4          1267.424                59.7
  2023             6.4        2.7          1394.580                61.3
  2024             7.5        1.2          1481.855                60.5
  2025             7.5        1.1          1633.921                57.3
  2026             7.0        2.0          1809.313                57.2

OK fmi_comparatif.csv → 63 lignes


In [5]:
print("=== COMPARATIF RÉGIONAL 2024 — DONNÉES CONFIRMÉES ===\n")
comp_2024 = (
    df_fmi_pivot[df_fmi_pivot['annee'] == 2024]
    .sort_values('croissance_pib', ascending=False)
)[['pays_nom','croissance_pib','inflation','pib_par_habitant']].round(2)
print(comp_2024.to_string(index=False))

print("\n=== TRAJECTOIRE 2024 → 2028 — CONFIRMÉ + PROJETÉ ===\n")
proj = df_fmi_pivot[df_fmi_pivot['annee'].isin([2024, 2025, 2026, 2027, 2028])].copy()

# Ajouter une colonne qui dit clairement le statut
def statut_annee(a):
    if a <= 2024: return 'confirmé'
    if a <= 2026: return 'estimé/en cours'
    return 'projection'

proj['statut'] = proj['annee'].apply(statut_annee)

pivot_proj = proj.pivot_table(
    index='pays_nom',
    columns='annee',
    values='croissance_pib'
).round(2)

print(pivot_proj.sort_values(2024, ascending=False).to_string())

=== COMPARATIF RÉGIONAL 2024 — DONNÉES CONFIRMÉES ===

    pays_nom  croissance_pib  inflation  pib_par_habitant
       Niger            10.3        9.1            707.47
       Bénin             7.5        1.2           1481.86
        Togo             6.3        2.9           1114.73
       Ghana             5.8       22.9           2419.25
Burkina Faso             4.8        4.2            981.97
     Nigeria             4.1       33.2           1083.51
    Cameroun             3.5        4.5           1830.40

=== TRAJECTOIRE 2024 → 2028 — CONFIRMÉ + PROJETÉ ===

annee         2024  2025  2026
pays_nom                      
Niger         10.3   6.9   6.7
Bénin          7.5   7.5   7.0
Togo           6.3   5.9   5.0
Ghana          5.8   6.0   4.8
Burkina Faso   4.8   5.0   4.9
Nigeria        4.1   4.0   4.1
Cameroun       3.5   3.1   3.3


In [6]:
df_fmi_pivot.to_csv(f'{LOCAL_PATH}/fmi_comparatif.csv', index=False)
df_fmi_pivot.to_csv(f'{DRIVE_PATH}/fmi_comparatif.csv', index=False)
print("OK fmi_comparatif.csv sauvegardé")

# Résumé final de toutes les sources
print("\n=== INVENTAIRE COMPLET DES DONNÉES ===\n")
sources = {
    'benin_events_clean.csv'  : '25 629 lignes — Signal médiatique GDELT',
    'comparatif_regional.csv' : '91 lignes     — Stabilité régionale GDELT',
    'benin_eco_events.csv'    : '10 079 lignes — Coopérations économiques GDELT',
    'benin_bilateral.csv'     : '104 lignes    — Relations bilatérales GDELT',
    'benin_media_bias.csv'    : '151 lignes    — Biais médiatique GDELT',
    'benin_sector_themes.csv' : '117 lignes    — Radar sectoriel GDELT/GKG',
    'fmi_comparatif.csv'      : '63 lignes     — Macro-économie FMI 2018-2026',
}

for fichier, description in sources.items():
    print(f"  {fichier:<35} {description}")

print("\n=== RÉPARTITION DES SOURCES ===\n")
total_gdelt = 25629 + 91 + 10079 + 104 + 151 + 117
total_complement = 63
total = total_gdelt + total_complement
print(f"  GDELT        : {total_gdelt:>6,} lignes  ({total_gdelt/total*100:.1f}%)")
print(f"  FMI          : {total_complement:>6,} lignes  ({total_complement/total*100:.1f}%)")
print(f"  TOTAL        : {total:>6,} lignes")
print(f"\n  Règle 'GDELT majoritaire' : {'OK' if total_gdelt/total > 0.5 else 'KO'} — {total_gdelt/total*100:.1f}% des données")

OK fmi_comparatif.csv sauvegardé

=== INVENTAIRE COMPLET DES DONNÉES ===

  benin_events_clean.csv              25 629 lignes — Signal médiatique GDELT
  comparatif_regional.csv             91 lignes     — Stabilité régionale GDELT
  benin_eco_events.csv                10 079 lignes — Coopérations économiques GDELT
  benin_bilateral.csv                 104 lignes    — Relations bilatérales GDELT
  benin_media_bias.csv                151 lignes    — Biais médiatique GDELT
  benin_sector_themes.csv             117 lignes    — Radar sectoriel GDELT/GKG
  fmi_comparatif.csv                  63 lignes     — Macro-économie FMI 2018-2026

=== RÉPARTITION DES SOURCES ===

  GDELT        : 36,171 lignes  (99.8%)
  FMI          :     63 lignes  (0.2%)
  TOTAL        : 36,234 lignes

  Règle 'GDELT majoritaire' : OK — 99.8% des données
